This notebook contains code accompanying our submission to NeurIPS 2024 Science of Deep Learning Workshop.

## 1. Setup 
(run, don't read!)

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import tqdm
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import MNIST, CIFAR10
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import mutual_info_score
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from IPython.display import clear_output
from collections import defaultdict
from itertools import islice
import random
import time
from pathlib import Path
import math

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

def randomseed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### (a) Data

In [2]:
dataset = 'MNIST' # 'MNIST' or 'CIFAR10'

if dataset == 'MNIST':
    transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307,), (0.3081,))])
    train_dataset = MNIST(root='.', train=True, download=True, transform=transform)
    test_dataset = MNIST(root='.', train=False, download=True, transform=transform)
elif dataset == 'CIFAR10':
    transform = torchvision.transforms.ToTensor()
    train_dataset = CIFAR10(root='.', train=True, download=True, transform=transform)
    test_dataset = CIFAR10(root='.', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

### (b) Models

In [3]:
# for MNIST

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 64, bias=False)
        self.fc2 = nn.Linear(64, 64, bias=False)
        self.fc3 = nn.Linear(64, 10, bias=False)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
# for CIFAR10

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(16 * 16 * 16, 64, bias=False)
        self.fc2 = nn.Linear(64, 64, bias=False)
        self.fc3 = nn.Linear(64, 10, bias=False)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = x.view(-1, 16 * 16 * 16)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
def new_model(dataset, device):
    if dataset == 'MNIST':
        model = MLP()
    elif dataset == 'CIFAR10':
        model = CNN()
    model = model.to(device)
    return model        

### (c) Evaluation

In [4]:
def accuracy(model, data):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data:
            outputs = model(images.to(device))
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels.to(device)).sum().item()
    return correct / total

def classwise_accuracy(model, data):
    model.eval()
    correct = defaultdict(int)
    total = defaultdict(int)
    with torch.no_grad():
        for images, labels in data:
            outputs = model(images.to(device))
            _, predicted = torch.max(outputs.data, 1)
            for i in range(len(labels)):
                label = labels[i].item()
                total[label] += 1
                correct[label] += int(predicted[i] == label)
    return [round(correct[i] / total[i], 3) if total[i] > 0 else 0 for i in range(10)]

def clusterability(model, cluster_U_indices, cluster_V_indices, num_clusters):
    A = model.fc2.weight ** 2
    mask = torch.zeros_like(A, dtype=torch.bool)
    
    for cluster_idx in range(num_clusters):
        u_indices = torch.tensor(cluster_U_indices[cluster_idx], dtype=torch.long)
        v_indices = torch.tensor(cluster_V_indices[cluster_idx], dtype=torch.long)
        mask[u_indices.unsqueeze(1), v_indices] = True
    
    intra_cluster_out_sum = torch.sum(A[mask])
    total_out_sum = torch.sum(A)
    
    return intra_cluster_out_sum / total_out_sum

## 2. Bipartite Spectral Graph Clustering

In [5]:
dataset, device

('MNIST', device(type='cuda', index=0))

In [61]:
unclustered_model = new_model(dataset, device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(unclustered_model.parameters(), lr=1e-3)
train_losses = []
# struct to store some gradients
grads = defaultdict(list)

In [62]:
randomseed(42)

for epoch in range(10):
    unclustered_model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = unclustered_model(data)
        loss = criterion(output, target)
        loss.backward()
        # store fc2 gradients
        for p in unclustered_model.named_parameters():
            if p[0] == 'fc2.weight':
                grads[batch_idx + (epoch * len(train_loader))].append(p[1].grad.clone())
        optimizer.step()
        train_losses.append(loss.item())
    acc = accuracy(unclustered_model, test_loader)
    print(f'Epoch {epoch+1}/{10}, Loss: {loss.item():.4f}, Accuracy: {acc:.4f}')

Epoch 1/10, Loss: 0.3162, Accuracy: 0.9419
Epoch 2/10, Loss: 0.0277, Accuracy: 0.9612
Epoch 3/10, Loss: 0.0941, Accuracy: 0.9674
Epoch 4/10, Loss: 0.1030, Accuracy: 0.9712
Epoch 5/10, Loss: 0.0616, Accuracy: 0.9719
Epoch 6/10, Loss: 0.1115, Accuracy: 0.9739
Epoch 7/10, Loss: 0.0070, Accuracy: 0.9691
Epoch 8/10, Loss: 0.0405, Accuracy: 0.9746
Epoch 9/10, Loss: 0.0003, Accuracy: 0.9760
Epoch 10/10, Loss: 0.0098, Accuracy: 0.9742


In [63]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=train_losses, mode='lines', name='', line=dict(color='darkred', width=2)))
fig.update_layout({'plot_bgcolor': 'rgba(255, 255, 255, 1)',})
fig.update_layout(
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='LightGray'),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='LightGray'),
)
fig.update_xaxes(title_text='Optimization Step')
fig.update_yaxes(title_text='CrossEntropy Loss')
fig.update_layout(width=600, height=400, autosize=False)
fig.show()

In [64]:
path = Path(f'results/{dataset}/')

In [65]:
# save the unclustered model
torch.save(unclustered_model.state_dict(), path/'unclustered_model.pth')


# load the unclustered model
unclustered_model = new_model(dataset, device)
unclustered_model.load_state_dict(torch.load(path/'unclustered_model.pth'))

<All keys matched successfully>

In [66]:
import numpy as np
import numpy as np
from sklearn.cluster import KMeans
from scipy.sparse.linalg import svds

def bipartite_spectral_clustering(similarity_matrix, k):
    
    A = similarity_matrix.detach().cpu().numpy()
    A = np.abs(A)
    
    D_U = np.diag(np.sum(A, axis=1))
    D_V = np.diag(np.sum(A, axis=0))

    D_U_inv_sqrt = np.linalg.inv(np.sqrt(D_U))
    D_V_inv_sqrt = np.linalg.inv(np.sqrt(D_V))

    A_tilde = D_U_inv_sqrt @ A @ D_V_inv_sqrt

    U, Sigma, Vt = svds(A_tilde, k=k)

    kmeans_U = KMeans(n_clusters=k, random_state=42).fit(U)
    kmeans_V = KMeans(n_clusters=k, random_state=42).fit(Vt.T)

    labels_U = kmeans_U.labels_
    labels_V = kmeans_V.labels_

    # convert labels to indices
    cluster_U_indices = defaultdict(list)
    cluster_V_indices = defaultdict(list)
    for i, label in enumerate(labels_U):
        cluster_U_indices[label].append(i)
    for i, label in enumerate(labels_V):
        cluster_V_indices[label].append(i)

    return cluster_U_indices, cluster_V_indices

In [67]:
if type(grads) is not torch.Tensor:
    grads = torch.stack(list({k: v[0] / v[0].norm() for k, v in grads.items()}.values()))

grads_similarity_matrix = torch.zeros(grads.shape[1], grads.shape[1]).to(device)

for i in range(grads.shape[0]):
    grads_similarity_matrix += torch.mm(grads[i].t(), grads[i])

similarity_matrices = [unclustered_model.fc2.weight, grads_similarity_matrix]
cluster_sizes = [2, 4, 6, 8, 10, 12, 14]
clusterability_scores = np.zeros((len(similarity_matrices), len(cluster_sizes)))

In [68]:
for i, similarity_matrix in enumerate(similarity_matrices):
    for num_clusters in cluster_sizes:
        cluster_U_indices, cluster_V_indices = bipartite_spectral_clustering(similarity_matrix, num_clusters)
        cl = clusterability(unclustered_model, cluster_U_indices, cluster_V_indices, num_clusters)
        clusterability_scores[i, cluster_sizes.index(num_clusters)] = cl.item()

In [69]:
# plot

fig = go.Figure()
fig.add_trace(go.Scatter(x=cluster_sizes, y=clusterability_scores[0], mode='lines+markers', name='Weight-based BSGC', line=dict(color='#A52A3D', width=3)))
fig.add_trace(go.Scatter(x=cluster_sizes, y=clusterability_scores[1], mode='lines+markers', name='Gradient-based BSGC', line=dict(color='#2A3DA5', width=3)))

fig.update_layout({'plot_bgcolor': 'rgba(255, 255, 255, 1)',})
fig.update_layout(
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='LightGray'),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='LightGray'),
)
# show x and y axis ticks
fig.update_xaxes(tickvals=cluster_sizes)
fig.update_yaxes(tickvals=np.arange(0, 1.1, 0.1))
# show zero gridlines
fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=False)
fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=False)
fig.update_xaxes(title_text='No. of Clusters (k)')
fig.update_yaxes(title_text='Clusterability')
fig.update_layout(width=500, height=500, autosize=False)

# y axis from 0 to 1
fig.update_yaxes(range=[0, 0.7])

# everthing latex font (for research paper)
fig.update_layout(font=dict(family='serif', size=20, color='black'))
fig.update_xaxes(title_font=dict(family='serif', size=20, color='black'))
fig.update_yaxes(title_font=dict(family='serif', size=20, color='black'))
fig.update_xaxes(tickfont=dict(family='serif', size=18, color='black'))
fig.update_yaxes(tickfont=dict(family='serif', size=18, color='black'))
fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=False)
fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=False)
# legend font
fig.update_layout(legend=dict(font=dict(family="serif", size=16, color='black')))

# show one horizontal line at y=0.25 with label "Random model (k=4)" at legend
fig.add_trace(go.Scatter(x=[2, 14], y=[0.25, 0.25], mode='lines', name='Random (k=4)', line=dict(color='black', width=3, dash='dash')))

# show legend at the top
fig.update_layout(legend=dict(orientation='h', x=0.32, y=1))

fig.show()

In [70]:
figures_path = Path('figures/mnist')

if not figures_path.exists():
    figures_path.mkdir()

In [71]:
# save as pdf

fig.write_image(str(figures_path / 'clusterability_scores.pdf'))

This doesn't seem to be good enough for interpretability.

## 3. Optimizing for Modularity

### (a) Train for a few steps.

In [72]:
dataset, device

('MNIST', device(type='cuda', index=0))

In [85]:
model = new_model(dataset, device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
train_losses = []
total_epochs = 10
initial_epochs = 0

randomseed(42)

for epoch in range(initial_epochs):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    acc = accuracy(model, test_loader)
    print(f'Epoch {epoch+1}/{total_epochs}, Loss: {loss.item():.4f}, Accuracy: {acc:.4f}')

### (b) Get Clusters (via BSGC)

In [87]:
num_clusters = 4
similarity_matrix = model.fc2.weight
cluster_U_indices, cluster_V_indices = bipartite_spectral_clustering(similarity_matrix, num_clusters)

for i in range(num_clusters):
    print(f'Cluster {i} has {len(cluster_U_indices[i])} nodes in U and {len(cluster_V_indices[i])} nodes in V')

Cluster 0 has 18 nodes in U and 14 nodes in V
Cluster 1 has 21 nodes in U and 20 nodes in V
Cluster 2 has 14 nodes in U and 16 nodes in V
Cluster 3 has 11 nodes in U and 14 nodes in V


In [86]:
clusterability_score = clusterability(model, cluster_U_indices, cluster_V_indices, num_clusters)
print(f'Clusterability score: {round(clusterability_score.item(), 3)}')

Clusterability score: 0.246


### (c) Train the rest of the model with the Enmeshment Loss

In [88]:
cluster_losses = []
ce_losses = []
lomda = 20.0

randomseed(42)

for epoch in range(total_epochs - initial_epochs):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss_ce = criterion(output, target)
        loss_cluster = clusterability(model, cluster_U_indices, cluster_V_indices, num_clusters)
        cluster_losses.append(loss_cluster.item())
        ce_losses.append(loss_ce.item())
        ###
        loss = loss_ce - lomda * loss_cluster
        ###
        loss.backward()
        optimizer.step()
    acc = accuracy(model, test_loader)
    print(f'Epoch {epoch+1}/{total_epochs}, Loss: {loss.item():.4f}, Clusterability: {loss_cluster.item():.4f}, Accuracy: {acc:.4f}')

Epoch 1/10, Loss: -19.7281, Clusterability: 0.9998, Accuracy: 0.9424
Epoch 2/10, Loss: -19.9616, Clusterability: 0.9997, Accuracy: 0.9585
Epoch 3/10, Loss: -19.9190, Clusterability: 0.9997, Accuracy: 0.9634
Epoch 4/10, Loss: -19.8533, Clusterability: 0.9997, Accuracy: 0.9655
Epoch 5/10, Loss: -19.9054, Clusterability: 0.9997, Accuracy: 0.9697
Epoch 6/10, Loss: -19.9487, Clusterability: 0.9997, Accuracy: 0.9670
Epoch 7/10, Loss: -19.9869, Clusterability: 0.9997, Accuracy: 0.9706
Epoch 8/10, Loss: -19.8546, Clusterability: 0.9998, Accuracy: 0.9698
Epoch 9/10, Loss: -19.9906, Clusterability: 0.9997, Accuracy: 0.9713
Epoch 10/10, Loss: -19.9873, Clusterability: 0.9998, Accuracy: 0.9724


In [89]:
# plot the two losses on side by side plots
fig = make_subplots(rows=1, cols=2, subplot_titles=('Cross Entropy Loss', 'Model Clusterability'))
fig.add_trace(go.Scatter(y=ce_losses, mode='lines', name='', line=dict(color='darkred', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(y=cluster_losses, mode='lines', name='', line=dict(color='darkblue', width=2)), row=1, col=2)
fig.update_layout({'plot_bgcolor': 'rgba(255, 255, 255, 1)',})
# show fine grid lines on both axes on both subplots
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', row=1, col=1)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', row=1, col=1)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', row=1, col=2)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', row=1, col=2)

fig.update_xaxes(title_text='Optimization Step', row=1, col=1)
fig.update_yaxes(title_text='Cross Entropy', row=1, col=1)
fig.update_xaxes(title_text='Optimization Step', row=1, col=2)
fig.update_yaxes(title_text='Clusterability', row=1, col=2)
fig.update_layout(width=1000, height=400, autosize=False)
fig.show()

In [78]:
model

MLP(
  (fc1): Linear(in_features=784, out_features=64, bias=False)
  (fc2): Linear(in_features=64, out_features=64, bias=False)
  (fc3): Linear(in_features=64, out_features=10, bias=False)
)

### (d) Store our clusters and the clustered model

In [79]:
path: str = f'results/{dataset}/'
Path(path).mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), f'{path}model.pth')
torch.save(cluster_U_indices, f'{path}cluster_U_indices.pth')
torch.save(cluster_V_indices, f'{path}cluster_V_indices.pth')

# also store the unclustered model
torch.save(unclustered_model.state_dict(), f'{path}unclustered_model.pth')

## 4. Interpreting the Clusters

In [80]:
path: str = f'results/{dataset}/'

# load the model and cluster indices
model = new_model(dataset, device)
model.load_state_dict(torch.load(f'{path}model.pth'))
cluster_U_indices = torch.load(f'{path}cluster_U_indices.pth')
cluster_V_indices = torch.load(f'{path}cluster_V_indices.pth')

In [90]:
classwise_accuracies = classwise_accuracy(model, test_loader)
classwise_accuracies

[0.987, 0.982, 0.981, 0.972, 0.948, 0.965, 0.986, 0.97, 0.961, 0.969]

In [82]:
num_clusters = len(cluster_U_indices)
num_clusters

4

### (a) Classwise accuracies with individual clusters turned ON and OFF

In [91]:
# plot the class-wise accuracies for each cluster turned OFF

num_clusters = len(cluster_U_indices)

classwise_accuracies_off = []
classwise_accuracies_on = []

for cluster_idx in tqdm.trange(num_clusters):
    model = new_model(dataset, device)
    model.load_state_dict(torch.load(path + 'model.pth'))
    model.to(device)
    
    # turn off the cluster
    for i in cluster_U_indices[cluster_idx]:
        model.fc2.weight.data[i] = 0
    for i in cluster_V_indices[cluster_idx]:
        model.fc2.weight.data[:, i] = 0
    
    classwise_accuracies_off.append(classwise_accuracy(model, test_loader))

model.load_state_dict(torch.load(path + 'model.pth'))

# plot the class-wise accuracies for each cluster turned ON

classwise_accuracies = []

for cluster_idx in tqdm.trange(num_clusters):
    model = new_model(dataset, device)
    model.load_state_dict(torch.load(path + 'model.pth'))
    model.to(device)
    
    # turn off every cluster except the current one
    for i in range(num_clusters):
        if i != cluster_idx:
            for j in cluster_U_indices[i]:
                model.fc2.weight.data[j] = 0
            for j in cluster_V_indices[i]:
                model.fc2.weight.data[:, j] = 0
    
    classwise_accuracies_on.append(classwise_accuracy(model, test_loader))

model.load_state_dict(torch.load(path + 'model.pth'))

100%|██████████| 4/4 [00:05<00:00,  1.36s/it]


<All keys matched successfully>

In [92]:
cifar_labels = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
mnist_labels = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [93]:
def plot_classwise_cluster_perf(num_clusters, classwise_accuracies_on, classwise_accuracies_off, cifar_labels=cifar_labels):

    colors = [
        # pc.qualitative.Dark24_r[0],
        '#2A3DA5',
        pc.qualitative.Dark24_r[15],
        # pc.qualitative.Dark24_r[5],
        '#A52A3D',
        pc.qualitative.Dark24_r[9],
    ]

    # Create subplots: one row per cluster
    fig = make_subplots(rows=2, cols=4, shared_xaxes=False, shared_yaxes=True, vertical_spacing=0.2, subplot_titles=[f'Cluster {i} (ON)' for i in range(num_clusters)] + [f'Cluster {i} (OFF)' for i in range(num_clusters)], row_heights=[0.15, 0.15])

    for i in range(num_clusters):
        fig.add_trace(go.Bar(
            x=cifar_labels if dataset == 'CIFAR10' else mnist_labels,
            y=classwise_accuracies_on[i],
            marker_color=colors[i],
            name=f'Cluster {i} (ON)',
        ), row=1, col=i+1)
        # x tick angle
        fig.update_xaxes(tickangle=0)

        fig.add_trace(go.Bar(
            x=cifar_labels if dataset == 'CIFAR10' else mnist_labels,
            y=classwise_accuracies_off[i],
            marker_color=colors[i],
            name=f'Cluster {i} (OFF)',
        ), row=2, col=i+1)

    # white background
    fig.update_layout(plot_bgcolor='white')

    # gridlines
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

    fig.update_layout(height=140 * num_clusters, width=1000)

    # hide the legend
    fig.update_layout(showlegend=True)

    # show y-axis title once in the exact middle of four subplots
    fig.add_annotation(
        text="Accuracy",
        xref="paper", yref="paper",
        x=-0.1, y=0.5,
        showarrow=False,
        font=dict(size=24, family="Computer Modern"),
        align="center",
        textangle=-90
    )
    # show x-axis title once in the exact middle of four subplots
    fig.add_annotation(
        text="Class",
        xref="paper", yref="paper",
        x=0.5, y=-0.15,
        showarrow=False,
        font=dict(size=22, family="Computer Modern"),
        align="center",
    )


    # latex font on everything (for research papers)

    fig.update_layout(
        font_family="Computer Modern",
        font_size=20,
    )

    # all y-axes have the same range and ticks
    fig.update_yaxes(range=[0, 1], tickvals=np.arange(0, 1, 0.2))

    # no legend
    fig.update_layout(showlegend=False)

    # show all x ticks
    fig.update_xaxes(tickvals=np.arange(10))

    # remove space from top of figure and add some space at the bottom
    # fig.update_layout(margin=dict(t=50, b=140))

    # everthing latex font (for research paper)
    fig.update_layout(font=dict(family='serif', size=20, color='black'))
    fig.update_xaxes(title_font=dict(family='serif', size=20, color='black'))
    fig.update_yaxes(title_font=dict(family='serif', size=20, color='black'))
    fig.update_xaxes(tickfont=dict(family='serif', size=18, color='black'))
    fig.update_yaxes(tickfont=dict(family='serif', size=18, color='black'))
    fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=False)
    return fig

fig = plot_classwise_cluster_perf(num_clusters, classwise_accuracies_on, classwise_accuracies_off)
fig.show()

In [94]:
# save as pdf

fig.write_image(str(figures_path / 'classwise_cluster_all_accuracies.pdf'), scale=5)

In [118]:
path

PosixPath('results/CIFAR10')

### (b) Layer Visualization

Note: This is not currently working (there's a bug), but skipping since it is also probably not that important.

In [103]:
path = Path(f'results/{dataset}/')

model = new_model(dataset, device)
model.load_state_dict(torch.load(f'{path}/model.pth'))
cluster_U_indices = torch.load(f'{path}/cluster_U_indices.pth')
cluster_V_indices = torch.load(f'{path}/cluster_V_indices.pth')

In [104]:
classwise_accuracy(model, test_loader)

[0.662, 0.716, 0.389, 0.547, 0.676, 0.359, 0.777, 0.625, 0.765, 0.714]

In [105]:
cluster_U_indices.keys(), cluster_V_indices.keys(), len(cluster_U_indices)

(dict_keys([3, 2, 0, 1]), dict_keys([1, 3, 2, 0]), 4)

In [106]:
def visualize_layer(layer):
    weights = layer.weight.detach().cpu().numpy() 
    num_outputs, num_inputs = weights.shape

    colors = pc.qualitative.G10

    node_x = []
    node_y = []
    node_text = []
    edge_x = []
    edge_y = []
    edge_color = []
    edge_width = []

    input_layer_positions = np.linspace(-1, 1, num_inputs)
    output_layer_positions = np.linspace(-1, 1, num_outputs)
    
    for i in range(num_inputs):
        node_x.append(input_layer_positions[i])
        node_y.append(-1)
        # node_text.append(f"{i+1}")
        node_text.append(".")

    for j in range(num_outputs):
        node_x.append(output_layer_positions[j])
        node_y.append(1)
        # node_text.append(f"{j+1}")
        node_text.append(".")

    # Add edges
    for i in range(num_inputs):
        for j in range(num_outputs):
            weight = weights[j, i]
            color = 'red' if weight < 0 else 'blue'
            # color based on cluster index
            
            thickness = abs(weight)
            edge_x.extend([input_layer_positions[i], output_layer_positions[j]])
            edge_y.extend([-1, 1])
            edge_color.append(color)
            edge_width.append(thickness)
            # edge_width.append(1)

    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=node_x, 
        y=node_y,
        mode='markers+text',
        text=node_text,
        textposition='top center',
        marker=dict(size=10, color='lightgray'),
        showlegend=False
    ))

    # also on bottom center
    fig.add_trace(go.Scatter(
        x=node_x, 
        y=[-1]*len(node_y),
        mode='markers+text',
        text=node_text,
        textposition='bottom center',
        marker=dict(size=10, color='lightgray'),
        showlegend=False
    ))
    
    for i in range(0, len(edge_x), 2):
        fig.add_trace(go.Scatter(
            x=[edge_x[i], edge_x[i+1]],
            y=[edge_y[i], edge_y[i+1]],
            mode='lines',
            line=dict(color=edge_color[i//2], width=edge_width[i//2]),
            showlegend=False
        ))

    # Update layout
    fig.update_layout(
        title=".",
        xaxis=dict(showgrid=False, zeroline=False),
        yaxis=dict(showgrid=False, zeroline=False, range=[-1.5, 1.5]),
        showlegend=False
    )

    # white background
    fig.update_layout(plot_bgcolor='white')
    # square aspect ratio
    fig.update_layout(
        autosize=False,
        width=1200,
        height=700,
    )

    # remove axis ticks
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)

    return fig

In [107]:
old_layer = model.fc2
old_layer.weight.shape

torch.Size([64, 64])

In [108]:
new_layer = nn.Linear(64, 64, bias=False)
new_weights = torch.zeros_like(new_layer.weight)
new_weights.shape

torch.Size([64, 64])

In [109]:
flat_U, flat_V = [], []
for i in range(len(cluster_U_indices)):
    flat_U.extend(cluster_U_indices[i])
    flat_V.extend(cluster_V_indices[i])

for i in range(len(flat_U)):
    for j in range(len(flat_V)):
        index_i = flat_U.index(i)
        index_j = flat_V.index(j)
        new_weights[index_j, index_i] = old_layer.weight[i, j]

new_layer.weight.data = new_weights

In [111]:
fig = visualize_layer(new_layer)
fig.show()

In [121]:
# save exact figure as shown in the notebook in high quality

fig.write_image(str(figures_path / 'clustered_layer.pdf'), scale=5)

In [166]:
# similarly rearrange fc3 layer
old_layer = model.fc3
old_layer.weight.shape

torch.Size([10, 64])

In [167]:
new_layer = nn.Linear(64, 10, bias=False)
new_weights = torch.zeros_like(new_layer.weight)
new_weights.shape

torch.Size([10, 64])

In [169]:
# here, we only have clusters on the input side

for i in range(len(flat_U)):
    for j in range(10):
        new_weights[j, i] = old_layer.weight[j, flat_U[i]]

new_layer.weight.data = new_weights

In [170]:
visualize_layer(new_layer).show()

### (c) Max-Activating Datapoints: Neuron Semanticity

This turns out to be (expectedly, kinda) not that helpful.

In [7]:
def max_activating_datapoints(k, model, data_loader, layer, neuron_index):
    # get the activations of the layer for all the data points
    activations = []
    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            flatten_images = images.view(-1, 28 * 28) if dataset == 'MNIST' else images
            flatten_images = F.max_pool2d(torch.relu(model.conv1(images)), 2).view(-1, 16 * 16 * 16) if dataset == 'CIFAR10' else flatten_images
            fc_1_out = torch.relu(model.fc1(flatten_images))
            fc_2_out = torch.relu(model.fc2(fc_1_out))
            fc_3_out = model.fc3(fc_2_out)
            if layer == 1:
                activations.extend(fc_1_out[:, neuron_index].cpu().numpy())
            elif layer == 2:
                activations.extend(fc_2_out[:, neuron_index].cpu().numpy())
            elif layer == 3:
                activations.extend(fc_3_out[:, neuron_index].cpu().numpy())

    activations = np.array(activations)
    indices = np.argsort(activations)[-k:]
    # print(indices)
    # get the images corresponding to the top k activations
    images = []
    labels = []
    for i in indices:
        images.append(data_loader.dataset[i][0])
        labels.append(data_loader.dataset[i][1])
    return images, labels

In [112]:
for neuron_index in range(len(model.fc2.weight))[:10]:
    _, labels = max_activating_datapoints(20, model, test_loader, 2, neuron_index)
    print(f"Neuron {neuron_index} is most activated by the following digits: {labels}")

Neuron 0 is most activated by the following digits: [8, 0, 2, 3, 6, 6, 0, 2, 4, 2, 6, 5, 6, 2, 2, 0, 2, 5, 2, 5]
Neuron 1 is most activated by the following digits: [0, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 7, 7, 7, 7]
Neuron 2 is most activated by the following digits: [7, 1, 7, 7, 9, 9, 9, 1, 7, 0, 1, 7, 7, 9, 7, 7, 1, 9, 0, 7]
Neuron 3 is most activated by the following digits: [8, 8, 8, 1, 0, 0, 1, 8, 9, 8, 0, 8, 8, 9, 8, 9, 1, 8, 8, 8]
Neuron 4 is most activated by the following digits: [2, 4, 7, 7, 1, 6, 4, 3, 4, 4, 7, 7, 1, 1, 1, 1, 1, 1, 0, 1]
Neuron 5 is most activated by the following digits: [4, 7, 9, 1, 5, 1, 3, 4, 3, 7, 7, 7, 7, 3, 4, 7, 7, 1, 7, 7]
Neuron 6 is most activated by the following digits: [6, 3, 4, 6, 2, 1, 6, 6, 6, 6, 6, 2, 1, 6, 6, 6, 6, 6, 6, 6]
Neuron 7 is most activated by the following digits: [6, 5, 2, 3, 3, 8, 8, 2, 8, 1, 3, 8, 3, 1, 5, 6, 7, 3, 3, 5]
Neuron 8 is most activated by the following digits: [1, 0, 9, 1, 1, 1, 1, 9, 1, 9, 1, 1, 1, 7, 9

In [8]:
def neuron_label_semanticity(model, test_loader, layer_id, k):
    neuron_semanticity = []
    n_semantic_neurons = [0 for _ in range(10)]
    for neuron_index in tqdm.trange(len(model.fc2.weight)):
        _, labels = max_activating_datapoints(k, model, test_loader, layer_id, neuron_index)
        # a rough sense of interpretability is the number of unique classes in the top k activations
        num_unique_classes = len(set(labels))
        neuron_semanticity.append(num_unique_classes / 10)
        n_semantic_neurons[num_unique_classes - 1] += 1
    
    return sum(neuron_semanticity) / len(neuron_semanticity), n_semantic_neurons

In [9]:
rsoil, dist = neuron_label_semanticity(model, test_loader, 2, 50)

100%|██████████| 64/64 [00:55<00:00,  1.15it/s]


## 5. Comparison with Unclustered Model

### (a) Neuron Label Semanticity

In [19]:
rsoil_unclustered, dist_unclustered = neuron_label_semanticity(unclustered_model, test_loader, 2, 50)

100%|██████████| 64/64 [00:55<00:00,  1.15it/s]


In [21]:
# aggregate sum of dist till index i
dist_agg = [0] * 10
for i in range(10):
    dist_agg[i] = sum(dist[i:])

dist_unclustered_agg = [0] * 10
for i in range(10):
    dist_unclustered_agg[i] = sum(dist_unclustered[i:])

In [22]:
# plot the distribution of the number of unique classes in the top k activations for each neuron in the fc2 layer of the clustered model against the unclustered model

fig = go.Figure()

# lines for the clustered model, not histogram
fig.add_trace(go.Scatter(
    x=list(range(10)),
    y=dist_agg,
    mode='lines',
    name='Clustered model',
    line=dict(color='darkblue', width=2)
))

# lines for the unclustered model, not histogram
fig.add_trace(go.Scatter(
    x=list(range(10)),
    y=dist_unclustered_agg,
    mode='lines',
    name='Unclustered model',
    line=dict(color='darkred', width=2)
))

fig.update_layout(
    title_text='Distribution of the number of unique classes in the top k activations for each neuron in the fc2 layer',
    xaxis_title_text='Number of unique classes',
    yaxis_title_text='Number of neurons',
    barmode='overlay'
)

fig.update_traces(marker_line_width=0)

# white background
fig.update_layout(plot_bgcolor='white')

# gridlines

fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

fig.show()

### (b) Circuit discovery search-space reduction

In [23]:
total_params = sum(p.numel() for p in model.parameters())
total_params

267328

In [24]:
(3 * 16 * 3 * 3) + 16 +  4096 * 64 + 64 * 64 + 64 * 10

267328

In [25]:
trimmed_params = (3 * 16 * 3 * 3) + 16 +  4096 * 16 + 16 * 16 + 16 * 10

print(f'Percentage pruned: {(total_params - trimmed_params) / total_params * 100:.2f}%')

Percentage pruned: 75.16%


### (c) Circuit Complexity Reduction (CCR)

The reduction in the expected circuit size for a given behavior. Since we restrict circuit-forming to use fixed clusters, we can likely expect simpler circuits? For this we need to implement a circuit discovery method. For now, we'll just use accuracy-based pruning. We can shift to others (ACDC/EAP) later on.

In [95]:
def fast_label_perf(model, x, label):
    with torch.no_grad():
        output = model(x)
        criterion = nn.CrossEntropyLoss()
        target = torch.tensor([label] * x.size(0)).to(device)
        loss = criterion(output, target)
        accuracy = (output.argmax(dim=1) == target).sum().item() / x.size(0)
    return loss, accuracy

In [96]:
import copy

def prune_model(model, x, label, device, verbose=False):
    pruned_model = copy.deepcopy(model)
    
    layers = list(pruned_model.children())
    for layer in reversed(layers):
        if verbose:
            print(f'Pruning layer: {layer}')
        if isinstance(layer, nn.Linear):
            for neuron_idx in tqdm.trange(layer.weight.shape[0]):
                for weight_idx in range(layer.weight.shape[1]):
                    # Create a mask to zero out the weight
                    mask = torch.ones_like(layer.weight)
                    mask[neuron_idx, weight_idx] = 0
                    
                    # Apply the mask
                    original_weight = layer.weight[neuron_idx, weight_idx].item()
                    layer.weight.data[neuron_idx, weight_idx] = 0
                    
                    # Check performance
                    loss_pruned, acc_pruned = fast_label_perf(pruned_model, x, label)
                    loss_original, acc_original = fast_label_perf(model, x, label)
                    
                    # If performance decreases, restore the weight
                    if (loss_pruned - loss_original) > 0:
                        layer.weight.data[neuron_idx, weight_idx] = original_weight

        # fraction pruned
        num_zeros = torch.sum(layer.weight == 0).item()
        total_params = layer.weight.numel()
        if verbose:
            print(f'Fraction pruned: {num_zeros / total_params:.4f}')
                    
    return pruned_model

In [99]:
label = 8

label_data = torch.stack([test_dataset[i][0] for i in range(len(test_dataset)) if test_dataset[i][1] == label])

label_data = label_data.to(device)

label_data.shape

torch.Size([974, 1, 28, 28])

In [100]:
pruned_model = prune_model(model, label_data, label, device, verbose=True)

Pruning layer: Linear(in_features=64, out_features=10, bias=False)


100%|██████████| 10/10 [00:00<00:00, 19.81it/s]


Fraction pruned: 0.9719
Pruning layer: Linear(in_features=64, out_features=64, bias=False)


100%|██████████| 64/64 [00:03<00:00, 19.69it/s]


Fraction pruned: 0.9932
Pruning layer: Linear(in_features=784, out_features=64, bias=False)


100%|██████████| 64/64 [00:39<00:00,  1.62it/s]

Fraction pruned: 0.9771


In [101]:
# comparing with if we had pruned the unclustered model

pruned_model_unclustered = prune_model(unclustered_model, label_data, label, device, verbose=True)

Pruning layer: Linear(in_features=64, out_features=10, bias=False)


100%|██████████| 10/10 [00:00<00:00, 19.64it/s]


Fraction pruned: 0.9766
Pruning layer: Linear(in_features=64, out_features=64, bias=False)


100%|██████████| 64/64 [00:03<00:00, 19.41it/s]


Fraction pruned: 0.9905
Pruning layer: Linear(in_features=784, out_features=64, bias=False)


100%|██████████| 64/64 [00:39<00:00,  1.61it/s]

Fraction pruned: 0.9751


In [102]:
print(classwise_accuracy(pruned_model, test_loader))
print(classwise_accuracy(pruned_model_unclustered, test_loader))

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]


In [103]:
def effective_circuit_size(model):
    # fraction of non-zero weights
    total_params = 0
    num_zeros = 0
    for layer in model.children():
        if isinstance(layer, nn.Linear):
            total_params += layer.weight.numel()
            num_zeros += torch.sum(layer.weight == 0).item()
    return round(1 - (num_zeros / total_params), 3)

In [104]:
effective_circuit_size(model), effective_circuit_size(pruned_model), effective_circuit_size(pruned_model_unclustered)

(1.0, 0.022, 0.024)

In [52]:
path

'results/CIFAR10/'

In [63]:
# store the unclustered model
torch.save(unclustered_model.state_dict(), f'{path}unclustered_model.pth')

In [105]:
# effective circuit sizes for pruned models v pruned unclustered models for each label
# NOTE:
# this will take a while to run; better to run this as a script in the background (there's a "pruning.py" for this)
# and then directly load the results

ecs_pruned_all_labels = []
ecs_pruned_unclustered_all_labels = []

for label in tqdm.trange(10):
    label_data = torch.stack([test_dataset[i][0] for i in range(len(test_dataset)) if test_dataset[i][1] == label])
    label_data = label_data.to(device)
    
    pruned_model = prune_model(model, label_data, label, device, verbose=False)
    pruned_model_unclustered = prune_model(unclustered_model, label_data, label, device, verbose=False)
    
    ecs_pruned_all_labels.append(effective_circuit_size(pruned_model))
    ecs_pruned_unclustered_all_labels.append(effective_circuit_size(pruned_model_unclustered))

    # print the effective circuit sizes for each label
    print(f'Label: {label}, ECS (pruned): {ecs_pruned_all_labels[-1]}, ECS (pruned unclustered): {ecs_pruned_unclustered_all_labels[-1]}')

torch.save(ecs_pruned_all_labels, path + 'ecs_pruned_all_labels.pth')
torch.save(ecs_pruned_unclustered_all_labels, path + 'ecs_pruned_unclustered_all_labels.pth')

  0%|          | 0/10 [00:18<?, ?it/s]


KeyboardInterrupt: 

In [106]:
path = 'results/MNIST/'

# load the effective circuit sizes
ecs_pruned_all_labels = torch.load(path + 'ecs_pruned_all_labels.pth')
ecs_pruned_unclustered_all_labels = torch.load(path + 'ecs_pruned_unclustered_all_labels.pth')

In [110]:
ecs_pruned_all_labels, ecs_pruned_unclustered_all_labels

([0.013, 0.015, 0.016, 0.01, 0.014, 0.027, 0.022, 0.012, 0.022, 0.014],
 [0.026, 0.021, 0.033, 0.031, 0.024, 0.03, 0.027, 0.03, 0.024, 0.023])

In [111]:
type(ecs_pruned_all_labels[0])

float

In [50]:
# cifar label names
cifar_labels = ['airplane', 'auto.', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
mnist_labels = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [120]:
fig = go.Figure()

# Calculate percentage increase
percentage_increase = [(unclustered - clustered) / unclustered * 100 
                       for clustered, unclustered in zip(ecs_pruned_all_labels, ecs_pruned_unclustered_all_labels)]

fig.add_trace(go.Scatter(
    x=cifar_labels if dataset == 'CIFAR10' else mnist_labels,
    y=ecs_pruned_all_labels,
    mode='lines+markers',
    name='Pruned clustered model',
    line=dict(color='darkblue', width=2),
    fill='tonexty'  # This will shade the area between the two plots
))

fig.add_trace(go.Scatter(
    x=cifar_labels if dataset == 'CIFAR10' else mnist_labels,
    y=ecs_pruned_unclustered_all_labels,
    mode='lines+markers',
    name='Pruned unclustered model',
    line=dict(color='darkred', width=2),
    fill='tonexty'  # This will shade the area between the two plots
))

# thicker markers
fig.update_traces(marker=dict(size=10))

# Add percentage increase annotations
for i, (x, y, pct) in enumerate(zip(mnist_labels, ecs_pruned_unclustered_all_labels, percentage_increase)):
    fig.add_annotation(
        x=x,
        y=y,
        text=f"+{pct:.1f}%",
        showarrow=False,
        yshift=30,
        font=dict(
            size=18,
            color="black",
            family='serif'
        ),
    )

# show a huge downward arrow from ecs_pruned_unclustered_all_labels to ecs_pruned_all_labels for each label
# for i, (x, y1, y2) in enumerate(zip(mnist_labels, ecs_pruned_unclustered_all_labels, ecs_pruned_all_labels)):
#     fig.add_annotation(
#         x=x,
#         y=(y1 + y2) / 2,
#         text="",
#         showarrow=True,
#         arrowhead=2,
#         arrowsize=float(0.2+ abs(y1 - y2) / max(ecs_pruned_unclustered_all_labels) * 2),
#         arrowwidth=2,
#         arrowcolor='darkred',
#         ax=0,
#         ay=-50,
#     )

fig.update_layout(
    title_text='',
    xaxis_title_text='Label',
    yaxis_title_text='Effective Circuit Size (lesser is better)',
    width=800,  # Fixed width
    height=600,  # Fixed height
    plot_bgcolor='white',
    yaxis=dict(range=[0, max(max(ecs_pruned_all_labels), max(ecs_pruned_unclustered_all_labels)) * 1.1]),  # y-axis starts at 0
)

fig.update_xaxes(
    tickmode='linear',
    tick0=0,
    dtick=1,
    showgrid=True,
    gridwidth=1,
    gridcolor='LightGray'
)

fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

# fig width and height
fig.update_layout(
    autosize=False,
    width=1000,
    height=550,
)

# increase font size of x ticks
fig.update_xaxes(tickfont=dict(size=14))
# make them at 45 degree angle
fig.update_xaxes(tickangle=0)

# legend on top
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.02,
    xanchor="left",
    x=0
))

# everthing latex font (for research paper)
fig.update_layout(font=dict(family='serif', size=20, color='black'))
fig.update_xaxes(title_font=dict(family='serif', size=20, color='black'))
fig.update_yaxes(title_font=dict(family='serif', size=20, color='black'))
fig.update_xaxes(tickfont=dict(family='serif', size=20, color='black'))
fig.update_yaxes(tickfont=dict(family='serif', size=18, color='black'))
fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=False)

fig.show()

Beautiful! 🤌

In [122]:
# save as pdf
figures_path = Path('figures/mnist')

fig.write_image(str(figures_path / 'ecs_pruned_all_labels.pdf'))
# fig.write_image(str(figures_path / 'ecs_pruned_all_labels.jpg'), scale=5)

In [15]:
print(classwise_accuracy(unclustered_model, test_loader))

[0.704, 0.75, 0.402, 0.466, 0.655, 0.499, 0.78, 0.669, 0.683, 0.716]


In [17]:
print(classwise_accuracy(model, test_loader))

[0.662, 0.716, 0.389, 0.547, 0.676, 0.359, 0.777, 0.625, 0.765, 0.714]


In [69]:
fig = make_subplots(rows=2, cols=5, subplot_titles=[f'Label {i}' for i in range(10)])

for i in range(10):
    img, _ = test_dataset[i]  # Assuming test_dataset returns a tuple (image, label)
    img = img.permute(1, 2, 0).cpu().numpy()  # Change shape from (C, H, W) to (H, W, C)
    img = (img * 255).astype(np.uint8)  # Ensure pixel values are in the range [0, 255]
    
    fig.add_trace(go.Image(z=img), row=(i // 5) + 1, col=(i % 5) + 1)

fig.update_layout(height=600, width=1000, title_text='One image for each label in CIFAR10')
fig.show()

Thank you! Feel free to contact Satvik (zsatvik@gmail.com) for stuff related to this codebase.